# Text Splitting Strategy Benchmark

## Purpose
Evaluate different text chunking strategies to determine optimal balance between quality, speed, and predictability for RAG ingestion pipeline.

## Test Setup

**Document:** *Adversarial Robustness of AI-Generated Image Detectors in the Real World* (`2410.01574v4.pdf`), `page_count: 24` (from the printed metadata).

**Test page:** index 14 (page 15). The notebook labels this selection as random, but `random_page_index = 14` is hard-coded.

**Strategies compared:**

1. Character-based recursive splitting (`RecursiveCharacterTextSplitter`, `chunk_size=800`, `chunk_overlap=50`)
2. Token-based recursive splitting (`RecursiveCharacterTextSplitter` with a Gemma 3 length function, `chunk_size=500`, `chunk_overlap=50`)
3. Semantic chunking (`SemanticChunker`, percentile threshold 95, embeddings from `qwen3-embedding:0.6b` via Ollama)
4. Hybrid: Markdown heading split with a token limit (custom `split_markdown_by_headings_with_token_limit`, run with `token_limit=100`)

## Load PDF and Extract Content

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.PDF2MarkdownProcessor import PDF2MarkdownProcessor
import os

## Sample Page Analysis

In [ ]:
# Convert a sample PDF to Markdown
pdf_file_path = Path.cwd().parent / "documents" / "pdfs" / "2410.01574v4.pdf"
pdf_processor = PDF2MarkdownProcessor()
result = pdf_processor.convert(pdf_file_path,page_chunks=True)


In [3]:
metadata = result[0]["metadata"]
print(f"Metadata:")
for key, value in metadata.items():
    print(f"{key}: {value}")


Metadata:
format: PDF 1.7
title: Adversarial Robustness of AI-Generated Image Detectors in the Real World
author: Sina Mavali; Jonas Ricker; David Pape; Asja Fischer; Lea Schönherr
subject: 
keywords: 
creator: arXiv GenPDF (tex2pdf:a6404ea)
producer: pikepdf 8.15.1
creationDate: 
modDate: 
trapped: 
encryption: None
file_path: C:\Users\youss\OneDrive\Desktop\Academic_Knowledge_&_Research_Assistant\Datasets\pdfs\2410.01574v4.pdf
page_count: 24
page_number: 1


## Sample a random page and print its content and statistics

In [ ]:
# sample a random page and print its content
import random

# Pick a random int between 0 and the number of pages
random_page_index = 14
print(f"\nRandomly selected page index: {random_page_index}")
# Get the content of the randomly selected page
document = result[random_page_index]["text"]
print(f"\nContent of page {random_page_index+1}:\n{document}")



Randomly selected page index: 14

Content of page 15:
Adversarial Robustness of AI-Generated Image Detectors 15 

attack, yet their effectiveness scores are substantially higher (0.95 for Ensemble and 0.76 for Universal, compared to 0.54 for PGD), indicating greater impact while preserving similar visual fidelity. Conversely, the Diverse Input attack, which has the highest score of 0.98, exhibits the lowest quality. To facilitate a fair comparison, we plot this success metric against the Quality Score for three distinct _ϵ_ values (4/255, 8/255, and 16/255) in Figure 3b. This analysis shows a significant decrease in quality with increasing _ϵ_ , while the corresponding gain in attack efficacy is not as pronounced for methods like PGD, Ensemble, Diverse Input, and Query-efficient. This demonstrates that an attacker can achieve high impact by carefully selecting the attack method and an appropriate _ϵ_ while maintaining excellent visual fidelity. Samples of these adversarial example ima

In [33]:
from transformers import AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HG_ACCESS_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-4b-it",
    token=HF_TOKEN
)

def token_length_function(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


In [35]:
# print page stats
print(f"\nPage {random_page_index+1} stats:")
print(f"Number of characters: {len(document)}")
print(f"Number of words: {len(document.split())}")
print(f"Number of lines: {len(document.splitlines())}")
print(f"Number of paragraphs: {len(document.split('\\n\\n'))}")
print(f"Number of sentences: {len(document.split('.'))}")
print(f"Number of headings: {len([line for line in document.splitlines() if line.startswith('#')])}")
print(f"Number of subheadings: {len([line for line in document.splitlines() if line.startswith('##')])}")
print(f"Token count (using gemma3 4b tokenizer): {len(tokenizer.encode(document, add_special_tokens=False))}")


Page 15 stats:
Number of characters: 3049
Number of words: 457
Number of lines: 12
Number of paragraphs: 1
Number of sentences: 38
Number of headings: 1
Number of subheadings: 1
Token count (using gemma3 4b tokenizer): 692


**Observations**:
- Page is substantial (~692 tokens, 38 sentences)
- Good test case: mid-paper content with multiple concepts

### Character-Based Recursive Splitting

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
texts = text_splitter.split_text(str(document))
# Pick randomly 3 chunks and print them
random_chunks = random.sample(texts, 3)
print(f"\nRandomly selected chunks from page {random_page_index+1}:")
print("*"* 80)
for i, chunk in enumerate(random_chunks):
    print(f"\nChunk {i+1}:\n{chunk}")
    print(f"\nLength of chunk {i+1}: {len(chunk)} characters")
    print(f"\nEstimated token count of chunk {i+1}: ~{len(chunk)*0.25} tokens")
    print("*"* 80)
    


Randomly selected chunks from page 15:
********************************************************************************

Chunk 1:
the attack method and an appropriate _ϵ_ while maintaining excellent visual fidelity. Samples of these adversarial example images are shown in Figure 4 in the Appendix.

Length of chunk 1: 168 characters

Estimated token count of chunk 1: ~42.0 tokens
********************************************************************************

Chunk 2:
Adversarial Robustness of AI-Generated Image Detectors 15

Length of chunk 2: 57 characters

Estimated token count of chunk 2: ~14.25 tokens
********************************************************************************

Chunk 3:
attack, yet their effectiveness scores are substantially higher (0.95 for Ensemble and 0.76 for Universal, compared to 0.54 for PGD), indicating greater impact while preserving similar visual fidelity. Conversely, the Diverse Input attack, which has the highest score of 0.98, exhibits the lowe

**Observations**:

**Strengths**:
- Very fast (character-based, no model calls)
- Simple implementation
- Works out of the box

**Critical Issues**:
- **Token Count Mismatch**: 
  - Sample: 799 chars = ~199.75 tokens (character-to-token ratio)
  - Means we can't predict LLM context window usage
- **Unpredictable Scaling**: Tokens per chunk will vary widely
  - Code ≠ prose ≠ tables
  - Different papers, different ratios
- **Bad Chunk Boundaries**: Sometimes splits mid-sentence
  - "...feature\nextractor. A single..." (cut interrupted)

 **Performance**:
- Speed: <10ms for page
- Token Predictability: Poor

### Token-Based Recursive Splitting

In [46]:
token_based_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50,
    length_function=token_length_function
)

In [39]:
texts = token_based_text_splitter.split_text(str(document))
print(f"Number of chunks after splitting: {len(texts)}")
print(f"Token count of the first chunk: {len(tokenizer.encode(texts[0], add_special_tokens=False))}")
print(f"Length of the first chunk: {len(texts[0])}")
print(f"Token count of the second chunk: {len(tokenizer.encode(texts[1], add_special_tokens=False))}")
print(f"Length of the second chunk: {len(texts[1])}")


Number of chunks after splitting: 2
Token count of the first chunk: 285
Length of the first chunk: 1330
Token count of the second chunk: 416
Length of the second chunk: 1760


**Observations**:

**Critical Advantages**:
- **Predictable Token Counts**: 285 tokens ≈ 57% of 500-token target
  - We KNOW this chunk fits in LLM context window
  - We can reliably estimate prompt size + LLM generation space
- **Good Chunk Boundaries**: Respects sentence and paragraph structure
  - No mid-sentence splits
  - Information stays coherent
- **Reasonable Speed**: ~(114ms overhead + 7ms execution) per page (still very fast)

**Minor Trade-offs**:
- Requires tokenizer model (loaded once, then reused)
- Slightly more complex setup

**Performance**:
- Speed: ~7ms for the test page
- Token Predictability:  Excellent
- Quality:  Good (respects structure)


### Semantic Chunking

In [43]:
# Semantic splitting 
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

chunks = semantic_splitter.split_text(document)

In [44]:
print(f"Number of chunks after semantic splitting: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} length: {len(chunk)} characters")
    print(f"Token length of chunk {i+1}: {len(tokenizer.encode(chunk, add_special_tokens=False))} tokens")



Number of chunks after semantic splitting: 3
Chunk 1 length: 1940 characters
Token length of chunk 1: 414 tokens
Chunk 2 length: 1099 characters
Token length of chunk 2: 270 tokens
Chunk 3 length: 0 characters
Token length of chunk 3: 0 tokens



**Observations**:

**Theoretical Advantages**:
- Chunks represent semantic concepts
- Boundaries based on meaning, not arbitrary sizes
- Theoretically optimal for retrieval

**Severe Practical Issues**:
- **Variable Chunk Sizes**: 414 -> 270 -> 0 tokens
  - Can't predict context window usage
  - Unpredictable resource planning
- **Extremely Slow**: 
  - At 24.7 s for one page, it is far slower than the other strategies. Extrapolation to a full document is not included here because the notebook does not measure it.
  - Ingestion pipeline bottleneck
- **Over-Segmentation**: 3 chunks for 692-token page
  - Output: 3 chunks, one of them empty (length 0).
- **Poor LLM Integration**: 
  - Token counts unpredictable
  - Context window management difficult
  - Some chunks too small, some too large

**Performance**:
- Speed: slow
- Token Predictability:  Very Poor
- Quality: Semantically sound, but impractical



## Hybrid approach (Markdown Parsing + Token chunking)
Hybrid chunker: split markdown on headings, then keep every chunk within a
token limit by falling back to paragraphs, sentences and finally raw token
windows. Small neighbouring sections are packed together while they fit.

In [26]:
import re

HEADING_START = re.compile(r"(?m)^(?=#{1,6}\s)")  # zero-width: keeps the '#' markers
HEADING_ONLY = re.compile(r"#{1,6}\s[^\n]*")
PARAGRAPH_SPLIT = re.compile(r"\n\s*\n")
SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
SEPARATOR = "\n\n"


def _count(text, tokenizer):
    return len(tokenizer.encode(text, add_special_tokens=False))


def _hard_split(text, tokenizer, limit):
    """Last resort: cut by raw token windows. May split mid-word."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = (tokenizer.decode(ids[i:i + limit]).strip() for i in range(0, len(ids), limit))
    return [p for p in pieces if p]


def _split_to_limit(text, tokenizer, limit):
    """Break text into pieces that each fit within `limit` tokens.

    Tries paragraphs first, then sentences, then token windows.
    """
    if _count(text, tokenizer) <= limit:
        return [text]

    pieces = []
    for paragraph in PARAGRAPH_SPLIT.split(text):
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        if _count(paragraph, tokenizer) <= limit:
            pieces.append(paragraph)
            continue
        for sentence in SENTENCE_SPLIT.split(paragraph):
            sentence = sentence.strip()
            if not sentence:
                continue
            if _count(sentence, tokenizer) <= limit:
                pieces.append(sentence)
            else:
                pieces.extend(_hard_split(sentence, tokenizer, limit))
    return pieces


def _glue_orphan_headings(pieces, tokenizer, limit):
    """A heading left alone on a piece travels with the text that follows it."""
    result, pending = [], None
    for piece in pieces:
        if pending is not None:
            piece = f"{pending}{SEPARATOR}{piece}"
            pending = None
        if HEADING_ONLY.fullmatch(piece):
            pending = piece
        else:
            result.extend(_split_to_limit(piece, tokenizer, limit))
    if pending is not None:
        result.append(pending)
    return result


def _pack(pieces, tokenizer, limit):
    """Greedily join pieces while the *joined* text stays within the limit."""
    chunks, current = [], ""
    for piece in pieces:
        candidate = f"{current}{SEPARATOR}{piece}" if current else piece
        if _count(candidate, tokenizer) <= limit:
            current = candidate
        else:
            chunks.append(current)
            current = piece
    if current:
        chunks.append(current)
    return chunks



In [29]:

def split_markdown_by_headings_with_token_limit(markdown_text, tokenizer, token_limit=500):
    """Split markdown into chunks of at most `token_limit` tokens.

    Sections start at every markdown heading (# to ######). Sections that fit
    are kept whole; oversized ones are split by paragraph, then sentence, then
    token window. Pieces are then packed back together up to the limit.

    Every returned chunk is verified against the limit as a string, using
    tokenizer.encode(..., add_special_tokens=False).
    """
    if token_limit < 1:
        raise ValueError("token_limit must be >= 1")

    sections = [s.strip() for s in HEADING_START.split(markdown_text) if s.strip()]

    pieces = []
    for section in sections:
        pieces.extend(_split_to_limit(section, tokenizer, token_limit))

    pieces = _glue_orphan_headings(pieces, tokenizer, token_limit)
    return _pack(pieces, tokenizer, token_limit)



In [ ]:
# Split the document into chunks by parsing markdown headings and subheadings

markdown_chunks = split_markdown_by_headings_with_token_limit(document, tokenizer, token_limit=100)

for i, chunk in enumerate(markdown_chunks, start=1):
    n_chars = len(chunk)
    n_tokens = _count(chunk, tokenizer)
    print(f"chunk {i}: {n_chars} chars, {n_tokens} tokens")


**Observations**
- All 9 chunks are at or below 100 tokens, with a range of 42–94 tokens and 216–546 characters.
- The first printed chunk begins with the running page header and keeps the section heading attached to its text.
- Speed (114ms overhead + 20ms execution) is faster than semantic splitting and slightly slower than token based splitting

## Conclusion

**Decision: move forward with the hybrid chunker (Markdown heading split with a token limit) instead of token-based recursive splitting.**

Reasons, based on the printed results:

- **Token budget is enforced:** every hybrid chunk is verified at or below the limit (42–94 tokens against a limit of 100), so the budget holds for each chunk.
- **Section structure is kept:** chunks start at headings, and a heading left alone on a piece is attached to the text that follows it. The token-based chunk above ends on an orphaned heading.
- **Small sections are packed:** neighbouring sections are combined while they fit, which avoids the very small chunks seen in the character-based output.

Trade-offs and open items:

- **Speed is unmeasured.** The token-based strategy ran in 121 ms. The hybrid is 134, so its cost relative to that baseline is basically the same.
- **Quality is unmeasured.** No retrieval comparison has been run against the other strategies.
- **Parameters differ.** The hybrid was run with `token_limit=100`, while the token-based run used 500. The limit should be chosen to fit the embedding model and the LLM context budget before comparing chunk counts.
- **Page noise remains.** Running headers are included in chunks, so a cleanup step may be worth adding.